# 4. Финальная обработка патентной базы 2020–2025

Ноутбук берет готовую книгу **`3_patents_2020_2025_final.xlsx`**, использует шесть оригинальных CSV-реестров Роспатента по ранее указанным локальным путям и создает новую независимую папку:

`Выгрузка данных/4_Финальная патентная база 2020-2025/`

Обработка выполняет следующие операции:

- полностью удаляет объект со статусом `parse_validation_failed` по `patent_key`;
- сохраняет промышленные образцы, присваивая им `RU`, международность `0` и цитируемость `0`;
- добавляет **для всех шести типов объектов** страны регистрации, авторов и правообладателей;
- страна регистрации всех записей — `RU`, поскольку объекты взяты из российских государственных реестров;
- страна российского университета-правообладателя всегда учитывается как `RU`, иностранные соправообладатели добавляются из исходных полей;
- страны авторов извлекаются только из реально доступных полей Роспатента/готовой книги; при отсутствии сведений `RU` автоматически не приписывается;
- международность классического патента равна `1`, только если найдена зарубежная страна семейства или международный маршрут; иначе страна семьи `RU` и флаг `0`;
- МПК объединяется из оригинальных полей Роспатента и безопасно извлеченных полей Google Patents;
- создаются итоговый Excel, общий Parquet и контрольные листы.

Запустите **Run All**. Повторная интернет-выгрузка Google Patents не выполняется.

In [ ]:
# Установка только отсутствующих зависимостей
import importlib.util
import subprocess
import sys

packages = {
    'pandas': 'pandas',
    'numpy': 'numpy',
    'openpyxl': 'openpyxl',
    'xlsxwriter': 'xlsxwriter',
    'pyarrow': 'pyarrow',
    'pycountry': 'pycountry',
}
missing = [pkg for pkg, module in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    print('Устанавливаются:', ', '.join(missing))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
else:
    print('Все зависимости установлены.')

## Загрузка зафиксированного кода версии 4

Основной код зафиксирован по конкретному коммиту. Он сохраняется рядом с ноутбуком как `4_patent_finalizer_core.py`, после чего импортируется и выполняется.

In [ ]:
from pathlib import Path
from urllib.request import Request, urlopen

CORE_URL = (
    'https://raw.githubusercontent.com/pies112/mininv/'
    '27a97ec0ec00e94263094e4c0ef5276829f270ee/'
    '4_patent_finalizer_core.py'
)
CORE_PATH = Path.cwd() / '4_patent_finalizer_core.py'

request = Request(CORE_URL, headers={'User-Agent': 'Mozilla/5.0'})
with urlopen(request, timeout=60) as response:
    CORE_PATH.write_bytes(response.read())

print('Код версии 4 сохранен:', CORE_PATH)
print('Размер, КБ:', round(CORE_PATH.stat().st_size / 1024, 1))

In [ ]:
# Импорт и предварительная проверка путей
import importlib.util

spec = importlib.util.spec_from_file_location('patent_finalizer_v4', CORE_PATH)
patent_finalizer_v4 = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(patent_finalizer_v4)

input_file = patent_finalizer_v4.find_final_workbook()
print('Найдена итоговая книга версии 3:', input_file)
print('Исходные реестры Роспатента:')
for source in patent_finalizer_v4.SOURCE_SPECS:
    resolved = patent_finalizer_v4.resolve_macos_path(source.path)
    print(f'  {source.object_type}: {resolved} | существует={resolved.exists()}')

missing_sources = [
    source.object_type
    for source in patent_finalizer_v4.SOURCE_SPECS
    if not patent_finalizer_v4.resolve_macos_path(source.path).exists()
]
if missing_sources:
    raise FileNotFoundError('Не найдены реестры: ' + ', '.join(missing_sources))
print('Предварительная проверка пройдена.')

## Полный запуск

Эта ячейка читает готовую книгу версии 3 и оригинальные реестры, удаляет один невалидный объект, добавляет итоговые страновые и классификационные переменные и сохраняет результаты версии 4.

In [ ]:
outputs = patent_finalizer_v4.main()
outputs

In [ ]:
# Финальная проверка созданной книги
import pandas as pd

qc = pd.read_excel(outputs['excel'], sheet_name='4_QC_финал')
summary = pd.read_excel(outputs['excel'], sheet_name='4_Сводка_типы')
country_qc = pd.read_excel(outputs['excel'], sheet_name='4_Страны_QC')

display(qc)
display(summary)
display(country_qc)

failed = qc.loc[pd.to_numeric(qc['passed'], errors='coerce').fillna(0).eq(0)]
if len(failed):
    print('Есть проверки, требующие внимания:')
    display(failed)
else:
    print('Все обязательные автоматические проверки пройдены.')

print('Итоговый Excel:', outputs['excel'])
print('Итоговый Parquet:', outputs['parquet'])